
# CBECS 2018 — Method B: Energy Intensity + L1 + Actionable-Variable GA

This notebook implements the project workflow through the genetic-algorithm stage.

## Workflow

1. Compute the mean annual major-fuels consumption (`MFBTU`).
2. Keep buildings with `MFBTU` above that mean.
3. Define **energy intensity**:

\[
EI = \frac{MFBTU}{SQFT}
\]

4. Fit an **L1-regularized regression (LassoCV)** to the filtered buildings.
5. Keep an explicit whitelist of **actionable upgrade variables** separate from Lasso's selected predictive variables.
6. Refit the scenario model on the union of:
   - variables selected by Lasso, and
   - all available actionable variables.
7. Run a binary genetic algorithm over feasible upgrades.
8. Calculate each building's predicted efficiency improvement:

\[
\text{Efficiency \%}
=
100 \times
\frac{\widehat{EI}_{baseline}-\widehat{EI}_{upgrade}}
{\widehat{EI}_{baseline}}
\]

9. Report:
   - the **mean predicted efficiency improvement**, and
   - the **five buildings with the largest predicted efficiency percentages**.

### Interpretation

The GA produces **model-based counterfactual scenarios**, not causal engineering estimates. A variable that Lasso shrinks to zero is not automatically physically irrelevant; this notebook therefore keeps the intervention whitelist conceptually separate from feature selection.


In [1]:

from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20

CSV_PATH = Path("cbecs2018_final_public.csv")
CODEBOOK_PATH = Path("2018microdata_codebook.xlsx")



## Actionable-variable whitelist

These variables represent upgrades, controls, or technologies that are at least plausibly changeable by a building owner/operator.

The first group is the recommended **core GA set**. The second group is optional and can be enabled after reviewing the dataset and project scope.


In [2]:

CORE_ACTIONABLE_VARS = [
    "RENHVC",   # HVAC equipment upgrade
    "RENINS",   # insulation upgrade
    "RENWIN",   # window replacement
    "RENRFF",   # roof replacement
    "RENLGT",   # lighting upgrade
    "NWMNHT",   # main heating replaced
    "NWMNCL",   # main cooling replaced
    "EMCS",     # building automation system
    "ECN",      # economizer cycle
    "MAINT",    # regular HVAC maintenance
    "LED",      # LED lighting
    "SCHED",    # light scheduling
    "OCSN",     # occupancy sensors
    "DIM",      # multilevel lighting / dimming
    "DAYHARV",  # daylight harvesting
    "PLGCTRL",  # plug-load control
    "RFCOOL",   # cool roof
]

OPTIONAL_ACTIONABLE_VARS = [
    "SOPANEL",  # solar panels
    "SOTHERM",  # solar thermal
    "TINT",     # tinted window glass
    "REFL",     # reflective window glass
]

USE_OPTIONAL_ACTIONS = False

ACTIONABLE_VARS = CORE_ACTIONABLE_VARS.copy()
if USE_OPTIONAL_ACTIONS:
    ACTIONABLE_VARS += OPTIONAL_ACTIONABLE_VARS

print(f"Configured actionable variables: {len(ACTIONABLE_VARS)}")


Configured actionable variables: 17



## Load the microdata and codebook

The codebook is used to:
- verify labels,
- detect categorical response-code variables,
- and audit which actionable variables are actually present.

The numerical results require the CBECS microdata CSV.


In [3]:

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Microdata CSV not found at {CSV_PATH.resolve()}. "
        "Upload the 2018 CBECS microdata CSV or change CSV_PATH."
    )

data = pd.read_csv(CSV_PATH)

codebook = None
if CODEBOOK_PATH.exists():
    codebook = pd.read_excel(CODEBOOK_PATH, header=1)
    codebook.columns = [str(c).replace("\n", " ").strip() for c in codebook.columns]
    codebook = codebook.rename(columns={
        "Variable name": "variable",
        "Variable type": "variable_type",
        "Label": "label",
        "Values/Format codes": "format_codes",
        "Question text": "question_text",
    })

print(f"Microdata rows: {len(data):,}")
print(f"Microdata columns: {len(data.columns):,}")
display(data.head())


Microdata rows: 6,436
Microdata columns: 1,249


,PUBID,REGION,CENDIV,PBA,PUBCLIM,SQFT,SQFTC,WLCNS,RFCNS,RFCOOL,...,ZMFBTU,ZMFEXP,ZELCNS,ZELEXP,ZNGCNS,ZNGEXP,ZFKCNS,ZFKEXP,ZDHBTU,ZDHEXP
0,1,3,5,2,3,210000,8,1,4,2,...,0,0,0,0,9,9,1,1,0,0
1,2,4,9,2,4,28000,5,1,6,1,...,0,0,0,0,0,0,9,9,9,9
2,3,3,5,8,4,2100,2,1,4,2,...,0,0,0,0,9,9,9,9,9,9
3,4,3,7,5,5,240000,8,2,6,1,...,0,0,0,0,1,1,9,9,9,9
4,5,1,2,5,3,295000,8,3,6,2,...,0,0,0,0,0,0,9,9,9,9



## 1. Mean MFBTU and above-mean filter


In [4]:

for col in ["MFBTU", "SQFT"]:
    if col not in data.columns:
        raise KeyError(f"Required column {col!r} is missing from the CSV.")
    data[col] = pd.to_numeric(data[col], errors="coerce")

mean_mfbtu = data["MFBTU"].mean()

high_energy = data.loc[
    data["MFBTU"].gt(mean_mfbtu)
    & data["MFBTU"].notna()
    & data["SQFT"].gt(0)
].copy()

print(f"Mean MFBTU: {mean_mfbtu:,.2f} thousand Btu")
print(f"Buildings above mean: {len(high_energy):,}")


Mean MFBTU: 15,833,540.90 thousand Btu
Buildings above mean: 1,554



## 2. Method B target: energy intensity

`SQFT` and `MFBTU` are excluded from the predictor set by default because the target is directly constructed from them.


In [5]:

high_energy["ENERGY_INTENSITY"] = high_energy["MFBTU"] / high_energy["SQFT"]

display(
    high_energy[["MFBTU", "SQFT", "ENERGY_INTENSITY"]]
    .describe()
)


,MFBTU,SQFT,ENERGY_INTENSITY
count,1.554000e+03,1.554000e+03,1554.000000
mean,5.321970e+07,4.747459e+05,124.454633
std,6.471897e+07,4.256455e+05,82.358144
min,1.583440e+07,5.000000e+04,8.579058
25%,2.113559e+07,2.150000e+05,72.097956
50%,2.997132e+07,3.200000e+05,104.115334
75%,5.453573e+07,5.000010e+05,151.832213
max,8.025880e+08,2.100000e+06,923.315378



## 3. Audit actionable variables

For the core binary intervention variables in the codebook, `1 = Yes` and `2 = No`.  
The GA only modifies variables that are present in the CSV and have a feasible observed `No` state.


In [6]:

available_actionable = [v for v in ACTIONABLE_VARS if v in high_energy.columns]
missing_actionable = [v for v in ACTIONABLE_VARS if v not in high_energy.columns]

print("Actionable variables present in the CSV:")
print(available_actionable)

if missing_actionable:
    print("\nConfigured actionable variables not found:")
    print(missing_actionable)

if codebook is not None:
    audit = codebook.loc[
        codebook["variable"].isin(ACTIONABLE_VARS),
        ["variable", "label", "format_codes"]
    ].copy()
    display(audit)

if not available_actionable:
    raise ValueError("None of the configured actionable variables are present in the CSV.")


Actionable variables present in the CSV:
['RENHVC', 'RENINS', 'RENWIN', 'RENRFF', 'RENLGT', 'NWMNHT', 'NWMNCL', 'EMCS', 'ECN', 'MAINT', 'LED', 'SCHED', 'OCSN', 'DIM', 'DAYHARV', 'PLGCTRL', 'RFCOOL']


,variable,label,format_codes
9,RFCOOL,Derived variable: Cool roof,1=Yes\n2=No
27,RENRFF,Roof replacement,1=Yes\n2=No\nMissing=Not applicable
28,RENWIN,Window replacement,1=Yes\n2=No\nMissing=Not applicable
29,RENHVC,HVAC equipment upgrade,1=Yes\n2=No\nMissing=Not applicable
30,RENLGT,Lighting upgrade,1=Yes\n2=No\nMissing=Not applicable
33,RENINS,Insulation upgrade,1=Yes\n2=No\nMissing=Not applicable
293,NWMNHT,Main heating replaced,1=Yes\n2=No\nMissing=Not applicable
365,NWMNCL,Main cooling replaced,1=Yes\n2=No\nMissing=Not applicable
366,EMCS,Building automation system (BAS),1=Yes\n2=No\nMissing=Not applicable
376,ECN,Economizer cycle,1=Yes\n2=No\nMissing=Not applicable



## 4. Construct a leakage-controlled predictor set

The target is \(MFBTU/SQFT\), so the notebook excludes:

- `MFBTU`
- `SQFT`
- modeled end-use energy quantities
- direct expenditure variables
- imputation flags
- survey replicate weights
- the public-use building identifier

The explicit actionable whitelist is retained even if its variables would otherwise be removed by a broad text filter.


In [7]:

def build_feature_list(frame, codebook, actionable_vars):
    blocked_exact = {"MFBTU", "SQFT", "ENERGY_INTENSITY", "PUBID"}

    leakage_terms = (
        "modeled variable",
        "annual major fuels consumption",
        "expenditure",
        "imputation flag",
        "replicate weight",
        "adjusted weight",
    )

    label_lookup = {}
    if codebook is not None:
        label_lookup = (
            codebook.dropna(subset=["variable"])
            .drop_duplicates("variable")
            .set_index("variable")["label"]
            .astype(str)
            .to_dict()
        )

    kept = []

    for col in frame.columns:
        if col in blocked_exact:
            continue

        if col.startswith("Z") or col.startswith("FINALWT"):
            continue

        label = label_lookup.get(col, "").lower()
        leakage = any(term in label for term in leakage_terms)

        if col in actionable_vars:
            leakage = False

        if not leakage:
            kept.append(col)

    return kept


candidate_features = build_feature_list(
    high_energy,
    codebook,
    available_actionable,
)

print(f"Candidate raw predictors after leakage filtering: {len(candidate_features):,}")


Candidate raw predictors after leakage filtering: 575



## 5. Infer categorical versus numeric variables

Many CBECS columns are numeric response codes rather than continuous measurements.  
When the codebook contains explicit response mappings such as `1=Yes`, `2=No`, those fields are treated as categorical and one-hot encoded.


In [8]:

def infer_categorical_columns(feature_cols, frame, codebook):
    categorical = set()

    if codebook is not None:
        lookup = (
            codebook.dropna(subset=["variable"])
            .drop_duplicates("variable")
            .set_index("variable")["format_codes"]
            .astype(str)
            .to_dict()
        )

        for col in feature_cols:
            fmt = lookup.get(col, "")
            # Explicit response-code mappings usually contain "=".
            if "=" in fmt:
                categorical.add(col)

    # Also treat non-numeric columns as categorical.
    for col in feature_cols:
        if not pd.api.types.is_numeric_dtype(frame[col]):
            categorical.add(col)

    return sorted(categorical)


categorical_features = infer_categorical_columns(
    candidate_features,
    high_energy,
    codebook,
)

numeric_features = [
    c for c in candidate_features
    if c not in categorical_features
]

print(f"Categorical predictors: {len(categorical_features):,}")
print(f"Numeric predictors: {len(numeric_features):,}")


Categorical predictors: 570
Numeric predictors: 5



## 6. L1-regularized feature-selection model

Energy intensity is usually right-skewed, so the Lasso is trained on `log1p(ENERGY_INTENSITY)`.  
Predictions are transformed back to the original energy-intensity scale for reporting.

This Lasso stage is used to identify predictive variables. It is **not allowed to define the GA action space by itself**.



### Compatibility fix for pandas `pd.NA`

This notebook converts pandas nullable categorical values (`pd.NA`) to ordinary
`np.nan` before they enter scikit-learn. This avoids the error:

`TypeError: boolean value of NA is ambiguous`

that can occur with `SimpleImputer(strategy="most_frequent")` on some
pandas/scikit-learn combinations.


In [9]:

def sanitize_for_sklearn(frame, categorical_cols, numeric_cols):
    """
    Make pandas data safe for scikit-learn preprocessing.

    pandas' nullable StringDtype uses pd.NA. Some scikit-learn/pandas
    combinations can raise:
        TypeError: boolean value of NA is ambiguous
    during most-frequent imputation.

    This function:
      - converts categorical columns to plain Python-object dtype,
      - replaces pd.NA with np.nan,
      - coerces numeric columns to float with np.nan for invalid/missing values.
    """
    frame = frame.copy()

    for col in categorical_cols:
        s = frame[col].astype(object)
        frame[col] = s.where(pd.notna(s), np.nan)

    for col in numeric_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").astype(float)

    return frame


In [10]:

X = high_energy[candidate_features].copy()
y = high_energy["ENERGY_INTENSITY"].copy()

# Avoid pandas pd.NA inside scikit-learn imputers.
X = sanitize_for_sklearn(
    X,
    categorical_cols=categorical_features,
    numeric_cols=numeric_features,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

lasso_model = Pipeline([
    ("preprocess", preprocessor),
    ("lasso", LassoCV(
        cv=5,
        random_state=RANDOM_STATE,
        max_iter=100000,
        n_jobs=-1,
    )),
])

lasso_model.fit(X_train, np.log1p(y_train))

pred_test = np.expm1(lasso_model.predict(X_test))
pred_test = np.clip(pred_test, 0, None)

print(f"Selected alpha: {lasso_model.named_steps['lasso'].alpha_:.6g}")
print(f"Test MAE:  {mean_absolute_error(y_test, pred_test):,.4f} thousand Btu/sq ft")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, pred_test)):,.4f} thousand Btu/sq ft")
print(f"Test R²:   {r2_score(y_test, pred_test):.4f}")


/home/revaldi/Projects/Hackathon/Climate-Hacktion/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['BLRAIR_CO2' 'BLRAIR_OT2' 'BLRDUCT_CO2' 'BLRDUCT_OT2' 'BLRFNCL_CO2'
 'BLRFNCL_OT2' 'BLROTD_CO2' 'BLROTD_OT2' 'BLROTD_PR2' 'BLRRAD_CO2'
 'BLRRAD_OT2' 'BLRRAD_PR2' 'BLRRDFL_CO2' 'BLRRDFL_OT2' 'BLRRDFL_PR2'
 'BLRWTLP_CO2' 'BLRWTLP_OT2' 'BLRWTLP_PR2' 'BOILP_CO' 'BOILP_OT'
 'CHILLR_PR' 'CHILP_PR' 'FIREPLP_WO' 'FURNP_OT' 'FURNP_PR' 'FURNP_WO'
 'HTPHP_PR' 'LTZHRP' 'LZHRPC' 'OTCLEQ_PR' 'OTCLP_FK' 'OTCLP_NG' 'OTCLP_PR'
 'OTCLP_ST' 'OTHTP_CO' 'OTHTP_FK' 'OTHTP_PR' 'PKGHP_OT' 'SLFCNP_OT'
 'SLFCNP_WO']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


Selected alpha: 0.00312558
Test MAE:  35.4505 thousand Btu/sq ft
Test RMSE: 59.6180 thousand Btu/sq ft
Test R²:   0.4787


/home/revaldi/Projects/Hackathon/Climate-Hacktion/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['BLRAIR_CO2' 'BLRAIR_OT2' 'BLRDUCT_CO2' 'BLRDUCT_OT2' 'BLRFNCL_CO2'
 'BLRFNCL_OT2' 'BLROTD_CO2' 'BLROTD_OT2' 'BLROTD_PR2' 'BLRRAD_CO2'
 'BLRRAD_OT2' 'BLRRAD_PR2' 'BLRRDFL_CO2' 'BLRRDFL_OT2' 'BLRRDFL_PR2'
 'BLRWTLP_CO2' 'BLRWTLP_OT2' 'BLRWTLP_PR2' 'BOILP_CO' 'BOILP_OT'
 'CHILLR_PR' 'CHILP_PR' 'FIREPLP_WO' 'FURNP_OT' 'FURNP_PR' 'FURNP_WO'
 'HTPHP_PR' 'LTZHRP' 'LZHRPC' 'OTCLEQ_PR' 'OTCLP_FK' 'OTCLP_NG' 'OTCLP_PR'
 'OTCLP_ST' 'OTHTP_CO' 'OTHTP_FK' 'OTHTP_PR' 'PKGHP_OT' 'SLFCNP_OT'
 'SLFCNP_WO']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(



## 7. Recover raw variables selected by Lasso

One-hot encoded categories can produce multiple transformed features from a single raw CBECS variable.  
This cell maps the non-zero Lasso coefficients back to their original raw variable names.


In [11]:

transformed_names = lasso_model.named_steps["preprocess"].get_feature_names_out()
coefficients = lasso_model.named_steps["lasso"].coef_

coef_table = pd.DataFrame({
    "transformed_feature": transformed_names,
    "coefficient": coefficients,
})

coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
nonzero = (
    coef_table.loc[coef_table["coefficient"].ne(0)]
    .sort_values("abs_coefficient", ascending=False)
    .copy()
)


def transformed_to_raw(name):
    if name.startswith("num__"):
        return name[len("num__"):]

    if name.startswith("cat__"):
        remainder = name[len("cat__"):]
        # Match the longest categorical variable prefix.
        matches = [
            var for var in categorical_features
            if remainder == var or remainder.startswith(var + "_")
        ]
        if matches:
            return max(matches, key=len)

    return name


nonzero["raw_variable"] = nonzero["transformed_feature"].map(transformed_to_raw)

lasso_selected_raw = sorted(nonzero["raw_variable"].dropna().unique())

print(f"Non-zero transformed coefficients: {len(nonzero):,}")
print(f"Unique raw variables selected by Lasso: {len(lasso_selected_raw):,}")
display(nonzero.head(30))


Non-zero transformed coefficients: 186
Unique raw variables selected by Lasso: 119


,transformed_feature,coefficient,abs_coefficient,raw_variable
10274,cat__SQFTC_10,-0.566789,0.566789,SQFTC
9907,cat__SERVERC_4.0,0.540999,0.540999,SERVERC
10270,cat__SQFTC_6,0.426983,0.426983,SQFTC
8726,cat__PBA_5,-0.409493,0.409493,PBA
10273,cat__SQFTC_9,-0.226080,0.226080,SQFTC
8725,cat__PBA_4,0.214194,0.214194,PBA
8735,cat__PBA_16,0.210977,0.210977,PBA
8742,cat__PBA_91,0.207515,0.207515,PBA
10271,cat__SQFTC_7,0.199675,0.199675,SQFTC
8737,cat__PBA_18,-0.197988,0.197988,PBA



## 8. Force actionable variables into the scenario model

This is the deliberate manual-inclusion step.

The final scenario predictor set is:

`Lasso-selected raw variables ∪ actionable variables`

Therefore an actionable feature does not disappear merely because the feature-selection Lasso omitted it.

The final scenario model below is another LassoCV fit on this reduced union. This preserves the L1 framework while ensuring that all actionable variables are considered in the model specification.

**Important:** inclusion does not guarantee a non-zero coefficient. If the final Lasso still shrinks an actionable variable to zero, the current data/model do not support a detectable conditional association for that variable.


In [12]:

scenario_features = sorted(
    set(lasso_selected_raw)
    .union(available_actionable)
    .intersection(candidate_features)
)

scenario_categorical = [
    c for c in categorical_features
    if c in scenario_features
]
scenario_numeric = [
    c for c in scenario_features
    if c not in scenario_categorical
]

print(f"Scenario-model raw variables: {len(scenario_features):,}")
print(f"  Lasso-selected raw variables: {len(lasso_selected_raw):,}")
print(f"  Actionable variables forced into candidate set: {len(available_actionable):,}")


Scenario-model raw variables: 126
  Lasso-selected raw variables: 119
  Actionable variables forced into candidate set: 17


In [13]:

Xs = high_energy[scenario_features].copy()
ys = high_energy["ENERGY_INTENSITY"].copy()

Xs = sanitize_for_sklearn(
    Xs,
    categorical_cols=scenario_categorical,
    numeric_cols=scenario_numeric,
)

Xs_train = Xs.loc[X_train.index]
Xs_test = Xs.loc[X_test.index]
ys_train = ys.loc[y_train.index]
ys_test = ys.loc[y_test.index]

scenario_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), scenario_numeric),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), scenario_categorical),
])

scenario_model = Pipeline([
    ("preprocess", scenario_preprocessor),
    ("lasso", LassoCV(
        cv=5,
        random_state=RANDOM_STATE,
        max_iter=100000,
        n_jobs=-1,
    )),
])

scenario_model.fit(Xs_train, np.log1p(ys_train))

scenario_test_pred = np.expm1(scenario_model.predict(Xs_test))
scenario_test_pred = np.clip(scenario_test_pred, 0, None)

print(f"Scenario-model alpha: {scenario_model.named_steps['lasso'].alpha_:.6g}")
print(f"Scenario test MAE:  {mean_absolute_error(ys_test, scenario_test_pred):,.4f}")
print(f"Scenario test RMSE: {np.sqrt(mean_squared_error(ys_test, scenario_test_pred)):,.4f}")
print(f"Scenario test R²:   {r2_score(ys_test, scenario_test_pred):.4f}")


Scenario-model alpha: 0.00220504
Scenario test MAE:  35.6622
Scenario test RMSE: 59.4558
Scenario test R²:   0.4815



## 9. Genetic algorithm

The implementation below uses only NumPy/Pandas/scikit-learn; no GA library is required.

### Feasible mutation rule

For the configured binary upgrade variables:

- observed `1 = Yes` → keep `1`; the measure already exists
- observed `2 = No` → GA may keep `2` or change it to `1`
- missing or any other response → hold fixed

The GA never changes non-actionable features.

### Optional feasibility constraint

`MAX_NEW_UPGRADES` can limit the number of new interventions in a package.  
Set it to `None` for an unrestricted search.


In [18]:

GA_POPULATION = 10
GA_GENERATIONS = 50
GA_MUTATION_RATE = 0.1
GA_CROSSOVER_RATE = 0.85

# Example: set to 5 to allow at most five new measures per building.
MAX_NEW_UPGRADES = 5


def to_model_frame(row):
    frame = pd.DataFrame(
        [{col: row[col] for col in scenario_features}],
        columns=scenario_features,
    )

    return sanitize_for_sklearn(
        frame,
        categorical_cols=scenario_categorical,
        numeric_cols=scenario_numeric,
    )


def predict_intensity(row):
    log_pred = float(scenario_model.predict(to_model_frame(row))[0])
    return max(float(np.expm1(log_pred)), 0.0)


def eligible_upgrade_vars(row):
    eligible = []
    for var in available_actionable:
        if var not in scenario_features:
            continue

        value = row[var]

        try:
            is_no = pd.notna(value) and float(value) == 2.0
        except (TypeError, ValueError):
            is_no = False

        if is_no:
            eligible.append(var)

    return eligible


def genetic_optimize_building(
    row,
    population_size=GA_POPULATION,
    generations=GA_GENERATIONS,
    mutation_rate=GA_MUTATION_RATE,
    crossover_rate=GA_CROSSOVER_RATE,
    max_new_upgrades=MAX_NEW_UPGRADES,
    random_state=42,
):
    rng = np.random.default_rng(random_state)

    eligible = eligible_upgrade_vars(row)
    baseline_prediction = predict_intensity(row)

    if len(eligible) == 0:
        return row.copy(), baseline_prediction, baseline_prediction, []

    n_genes = len(eligible)

    def repair(individual):
        individual = individual.copy()

        if max_new_upgrades is not None and individual.sum() > max_new_upgrades:
            active = np.flatnonzero(individual)
            rng.shuffle(active)
            individual[active[max_new_upgrades:]] = 0

        return individual

    def decode(individual):
        candidate = row.copy()

        for bit, var in zip(individual, eligible):
            if bit == 1:
                candidate[var] = 1
            else:
                candidate[var] = 2

        return candidate

    def objective(individual):
        return predict_intensity(decode(individual))

    # First chromosome is always "no new upgrades" so the optimizer
    # cannot return a worse package simply because of stochastic search.
    population = np.zeros((population_size, n_genes), dtype=np.int8)

    for i in range(1, population_size):
        individual = rng.integers(0, 2, size=n_genes, dtype=np.int8)
        population[i] = repair(individual)

    best = population[0].copy()
    best_score = objective(best)

    for _ in range(generations):
        scores = np.array([objective(ind) for ind in population])

        order = np.argsort(scores)
        population = population[order]
        scores = scores[order]

        if scores[0] < best_score:
            best_score = float(scores[0])
            best = population[0].copy()

        elite_count = max(2, population_size // 5)
        elite = population[:elite_count]

        new_population = [ind.copy() for ind in elite]

        while len(new_population) < population_size:
            p1 = elite[rng.integers(0, elite_count)]
            p2 = elite[rng.integers(0, elite_count)]

            child = p1.copy()

            if n_genes > 1 and rng.random() < crossover_rate:
                point = rng.integers(1, n_genes)
                child[:point] = p1[:point]
                child[point:] = p2[point:]

            mutation_mask = rng.random(n_genes) < mutation_rate
            child[mutation_mask] = 1 - child[mutation_mask]

            child = repair(child)
            new_population.append(child)

        population = np.asarray(new_population, dtype=np.int8)

    optimized_row = decode(best)
    optimized_prediction = predict_intensity(optimized_row)

    selected_upgrades = [
        var for bit, var in zip(best, eligible)
        if bit == 1
    ]

    return (
        optimized_row,
        baseline_prediction,
        optimized_prediction,
        selected_upgrades,
    )



## 10. Run the GA on the held-out test buildings

Using the test set for this final scenario summary avoids reporting optimization results on the same observations used to fit the model.


In [19]:

ga_results = []

for position, idx in enumerate(Xs_test.index):
    row = high_energy.loc[idx].copy()

    (
        optimized_row,
        baseline_pred,
        optimized_pred,
        selected_upgrades,
    ) = genetic_optimize_building(
        row,
        random_state=RANDOM_STATE + position,
    )

    if baseline_pred > 0:
        efficiency_pct = (
            100.0
            * (baseline_pred - optimized_pred)
            / baseline_pred
        )
    else:
        efficiency_pct = np.nan

    ga_results.append({
        "row_index": idx,
        "PUBID": row["PUBID"] if "PUBID" in row.index else idx,
        "observed_energy_intensity": row["ENERGY_INTENSITY"],
        "baseline_pred_intensity": baseline_pred,
        "optimized_pred_intensity": optimized_pred,
        "efficiency_pct": efficiency_pct,
        "num_new_upgrades": len(selected_upgrades),
        "selected_upgrades": ", ".join(selected_upgrades),
    })

    if (position + 1) % 100 == 0:
        print(f"Optimized {position + 1:,} / {len(Xs_test):,} buildings")

ga_results_df = pd.DataFrame(ga_results)

print(f"Completed GA optimization for {len(ga_results_df):,} held-out buildings.")
display(ga_results_df.head())


Optimized 100 / 311 buildings
Optimized 200 / 311 buildings
Optimized 300 / 311 buildings
Completed GA optimization for 311 held-out buildings.


,row_index,PUBID,observed_energy_intensity,baseline_pred_intensity,optimized_pred_intensity,efficiency_pct,num_new_upgrades,selected_upgrades
0,6306,6307.0,89.588118,42.907551,41.845805,2.474496,2,"ECN, OCSN"
1,1141,1142.0,149.934656,116.312572,116.312572,0.000000,0,
2,1460,1461.0,108.826160,76.858300,76.858300,0.000000,0,
3,1461,1462.0,369.336434,222.727494,217.317452,2.428996,4,"RENHVC, SCHED, OCSN, PLGCTRL"
4,3792,3793.0,66.225859,55.858876,55.858876,0.000000,0,



# Final requested outputs

The mean below aggregates all valid building-level efficiency predictions.

The top-five table is sorted by **predicted efficiency percentage**, highest first.


In [20]:

valid = ga_results_df.dropna(subset=["efficiency_pct"]).copy()

mean_efficiency_prediction = valid["efficiency_pct"].mean()

print(
    "Mean predicted efficiency improvement across held-out buildings: "
    f"{mean_efficiency_prediction:.2f}%"
)

top5 = (
    valid
    .sort_values("efficiency_pct", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

print("\nFive samples with the largest predicted efficiency percentages:")
display(
    top5[
        [
            "PUBID",
            "observed_energy_intensity",
            "baseline_pred_intensity",
            "optimized_pred_intensity",
            "efficiency_pct",
            "num_new_upgrades",
            "selected_upgrades",
        ]
    ]
)


Mean predicted efficiency improvement across held-out buildings: 2.46%

Five samples with the largest predicted efficiency percentages:


,PUBID,observed_energy_intensity,baseline_pred_intensity,optimized_pred_intensity,efficiency_pct,num_new_upgrades,selected_upgrades
0,4724.0,66.518348,75.927953,69.207167,8.851531,4,"NWMNCL, SCHED, OCSN, RFCOOL"
1,1771.0,96.718912,103.070064,93.978016,8.821231,5,"NWMNCL, SCHED, OCSN, PLGCTRL, RFCOOL"
2,4280.0,91.342791,133.906375,122.120322,8.801712,4,"NWMNCL, OCSN, PLGCTRL, RFCOOL"
3,5017.0,134.809598,158.382249,144.457869,8.791629,5,"NWMNCL, ECN, OCSN, PLGCTRL, RFCOOL"
4,984.0,209.720025,241.773756,220.563904,8.772603,5,"NWMNCL, SCHED, OCSN, PLGCTRL, RFCOOL"



## 25% reduction target summary

This is a model-based scenario check against the project's 25% reduction objective.


In [21]:

valid["meets_25pct_target"] = valid["efficiency_pct"] >= 25.0

print(
    "Share of held-out buildings with >=25% predicted efficiency improvement: "
    f"{valid['meets_25pct_target'].mean():.1%}"
)

display(
    valid["efficiency_pct"]
    .describe()
    .to_frame("efficiency_pct")
)


Share of held-out buildings with >=25% predicted efficiency improvement: 0.0%


,efficiency_pct
count,311.000000
mean,2.455476
std,2.406756
min,0.000000
25%,0.000000
50%,2.451996
75%,3.819114
max,8.851531



## Methodological notes

1. **Manual inclusion is intentional.** The actionable whitelist is added to the final candidate set even when those variables were not selected by the first Lasso.
2. **A forced-in variable can still receive a zero Lasso coefficient.** That is preferable to manually forcing an effect size unsupported by the data.
3. **GA results are associative/counterfactual predictions, not causal savings.** CBECS is observational data.
4. **Historical renovation flags are proxies.** Variables such as `RENHVC`, `RENINS`, and `RENWIN` describe observed upgrade history and should not automatically be interpreted as engineering specifications for a future retrofit.
5. **The GA becomes more informative when constraints are added.** Future extensions can include upgrade cost, a maximum budget, mutually exclusive technologies, building-type restrictions, and embodied-carbon constraints.
6. **With a purely additive linear prediction model, optimization can be simpler than a GA.** The GA is retained because it provides a framework for later nonlinear models and real retrofit constraints.
